# Construire votre Réseau de Neurones Profond : Étape par Étape

Dans ce notebook, vous implémenterez toutes les fonctions nécessaires à la construction d'un réseau de neurones profond avec autant de couches que vous le souhaitez.

- Dans ce notebook, vous implémenterez toutes les briques fondamentales d'un réseau de neurones profond.
- Dans le notebook suivant, vous utiliserez ces fonctions pour construire un réseau profond appliqué à la classification d'images.

**À la fin de ce workshop, vous serez capable de :**

- Utiliser des unités non linéaires comme ReLU pour améliorer votre modèle
- Construire un réseau de neurones plus profond (avec plus d'une couche cachée)
- Implémenter une classe de réseau de neurones simple d'utilisation

**Notation** :
- L'exposant $[l]$ désigne une quantité associée à la $l^{	ext{ème}}$ couche.
- L'exposant $(i)$ désigne une quantité associée au $i^{	ext{ème}}$ exemple.
- L'indice $i$ désigne le $i^{	ext{ème}}$ élément d'un vecteur.

**Note sur l'évaluateur automatique :** Ce notebook contient des cellules évaluées automatiquement. Évitez d'ajouter des `print` supplémentaires, de modifier les paramètres des fonctions, ou d'utiliser des variables globales dans les exercices.

Commençons !

## Table des matières
- [1 - Packages](#1)
- [2 - Plan du notebook](#2)
- [3 - Initialisation](#3)
 - [3.1 - Réseau de neurones à 2 couches](#3-1)
 - [Exercice 1 - initialize_parameters](#ex-1)
 - [3.2 - Réseau de neurones profond à L couches](#3-2)
 - [Exercice 2 - initialize_parameters_deep](#ex-2)
- [4 - Module de propagation avant](#4)
 - [4.1 - Propagation linéaire avant](#4-1)
 - [Exercice 3 - linear_forward](#ex-3)
 - [4.2 - Propagation linéaire-activation avant](#4-2)
 - [Exercice 4 - linear_activation_forward](#ex-4)
 - [4.3 - Modèle à L couches](#4-3)
 - [Exercice 5 - L_model_forward](#ex-5)
- [5 - Fonction de coût](#5)
 - [Exercice 6 - compute_cost](#ex-6)
- [6 - Module de rétropropagation](#6)
 - [6.1 - Propagation linéaire arrière](#6-1)
 - [Exercice 7 - linear_backward](#ex-7)
 - [6.2 - Propagation linéaire-activation arrière](#6-2)
 - [Exercice 8 - linear_activation_backward](#ex-8)
 - [6.3 - Rétropropagation du modèle à L couches](#6-3)
 - [Exercice 9 - L_model_backward](#ex-9)
 - [6.4 - Mise à jour des paramètres](#6-4)
 - [Exercice 10 - Mettre à jour les paramètres](#ex-10)

<a name='1'></a>
## 1 - Bibliothèques

Commencez par importer tous les packages dont vous aurez besoin dans ce notebook.

- [numpy](www.numpy.org) est le package principal pour le calcul scientifique avec Python.
- [matplotlib](http://matplotlib.org) est une bibliothèque pour tracer des graphiques en Python.
- `dnn_utils` fournit quelques fonctions utiles pour ce notebook.
- `testCases` fournit des cas de test pour vérifier la correction de vos fonctions.
- `np.random.seed(1)` est utilisé pour assurer la reproductibilité des résultats. Ne le modifiez pas.

In [ ]:
### v1.2

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from testCases import *
from dnn_utils import sigmoid, sigmoid_backward, relu, relu_backward
from public_tests import *

import copy
%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)

<a name='2'></a>
## 2 - Plan du notebook

Pour construire votre réseau de neurones, vous allez implémenter plusieurs "fonctions auxiliaires". Ces fonctions seront utilisées dans le prochain notebook pour construire un réseau à deux couches et un réseau à L couches.

Chaque fonction auxiliaire est accompagnée d'instructions détaillées. Voici un aperçu des étapes de ce notebook :

- Initialiser les paramètres d'un réseau à deux couches et d'un réseau à L couches
- Implémenter le module de propagation avant (en violet dans la figure ci-dessous)
 - Compléter la partie LINÉAIRE de la propagation avant d'une couche (calcul de $Z^{[l]}$).
 - La fonction ACTIVATION (relu/sigmoid) est fournie.
 - Combiner ces deux étapes dans une nouvelle fonction [LINÉAIRE->ACTIVATION].
 - Empiler [LINÉAIRE->RELU] L-1 fois et ajouter [LINÉAIRE->SIGMOID] à la fin. Cela donne la fonction L_model_forward.
- Calculer la perte
- Implémenter le module de rétropropagation (en rouge dans la figure ci-dessous)
 - Compléter la partie LINÉAIRE de la rétropropagation d'une couche
 - Le gradient de la fonction ACTIVATION est fourni (relu_backward/sigmoid_backward)
 - Combiner ces deux étapes dans une nouvelle fonction [LINÉAIRE->ACTIVATION] arrière
 - Empiler [LINÉAIRE->RELU] arrière L-1 fois et ajouter [LINÉAIRE->SIGMOID] arrière dans L_model_backward
- Mettre à jour les paramètres

<img src="images/final outline.png" style="width:800px;height:500px;">
<caption><center><b>Figure 1</b></center></caption><br>

**Note** :

Pour chaque fonction de propagation avant, il existe une fonction de rétropropagation correspondante. C'est pourquoi à chaque étape du module avant, vous stockerez certaines valeurs dans un cache. Ces valeurs mises en cache sont utiles pour calculer les gradients lors de la rétropropagation.

<a name='3'></a>
## 3 - Initialisation

Vous allez écrire deux fonctions auxiliaires to initialiser le paramètres pour votre modèle. Le first fonction allez be used to initialiser paramètres pour un two couche modèle. Le second one generalizes this initialisation process to $L$ layers.

<a name='3-1'></a>
### 3.1 - Réseau de neurones à 2 couches

<a name='ex-1'></a>
### Exercice 1 - initialize_parameters

Create et initialiser le paramètres of le 2-couche neural network.

**instructions :**

- La structure du modèle is: *LINEAR -> RELU -> LINEAR -> SIGMOID*. - Utilisez cette initialisation aléatoire pour le weight matrices: `np.random.randn(d0, d1, ..., dn) * 0.01` avec le correct shape. Le documentation pour [np.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html)
- Utilisez zero initialisation pour le biases: `np.zeros(shape)`. Le documentation pour [np.zeros](https://numpy.org/doc/stable/reference/generated/numpy.zeros.html)

In [ ]:
# FONCTION : initialize_parameters
def initialize_parameters(n_x, n_h, n_y):
    """
 Argument:
 n_x -- taille de la couche d'entrée
 n_h -- taille de la couche cachée
 n_y -- taille de la couche de sortie
 
 Retourne :
 paramètres -- python dictionary containing votre paramètres:
 W1 -- weight matrix of shape (n_h, n_x)
 b1 -- bias vector of shape (n_h, 1)
 W2 -- weight matrix of shape (n_y, n_h)
 b2 -- bias vector of shape (n_y, 1)
 """
    
    np.random.seed(1)
    
    #(≈ 4 lines of code) # W1 = ... # b1 = ... # W2 = ... # b2 = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}
    
    return parameters    

In [ ]:
print("Test Case 1:\n")
parameters = initialize_parameters(3,2,1)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_1(initialize_parameters)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters(4,3,2)

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_test_2(initialize_parameters)

***Sortie attendue***
```
Cas de test 1:

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969 0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]
 All tests passed.

Cas de test 2:

W1 = [[ 0.01624345 -0.00611756 -0.00528172 -0.01072969]
 [ 0.00865408 -0.02301539 0.01744812 -0.00761207]
 [ 0.00319039 -0.0024937 0.01462108 -0.02060141]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[-0.00322417 -0.00384054 0.01133769]
 [-0.01099891 -0.00172428 -0.00877858]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='3-2'></a>
### 3.2 - Réseau de neurones profond à L couches

L'initialisation d'un réseau de neurones plus profond à L couches est plus complexe car il y a davantage de matrices de poids et de vecteurs de biais. Lorsque vous complétez la fonction `initialize_parameters_deep`, assurez-vous que les dimensions correspondent entre chaque couche. Rappelez-vous que $n^{[l]}$ est le nombre d'unités de la couche $l$. Par exemple, si la taille de votre entrée $X$ est $(12288, 209)$ (avec $m=209$ exemples), alors :

<table style="width:100%">
 <tr>
 <td> </td>  <td> <b>Shape of W</b> </td>  <td> <b>Shape of b</b> </td>  <td> <b>Activation</b> </td>
 <td> <b>Shape of Activation</b> </td>  <tr>
 <tr>
 <td> <b>couche 1</b> </td>  <td> $(n^{[1]},12288)$ </td>  <td> $(n^{[1]},1)$ </td>  <td> $Z^{[1]} = W^{[1]} X + b^{[1]} $ </td>  <td> $(n^{[1]},209)$ </td>  <tr>
 <tr>
 <td> <b>couche 2</b> </td>  <td> $(n^{[2]}, n^{[1]})$ </td>  <td> $(n^{[2]},1)$ </td>  <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td>  <td> $(n^{[2]}, 209)$ </td>  <tr>
 <tr>
 <td> $\vdots$ </td>  <td> $\vdots$ </td>  <td> $\vdots$ </td>  <td> $\vdots$</td>  <td> $\vdots$ </td>  <tr>  <tr>
 <td> <b>couche L-1</b> </td>  <td> $(n^{[L-1]}, n^{[L-2]})$ </td>  <td> $(n^{[L-1]}, 1)$ </td>  <td>$Z^{[L-1]} = W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td>  <td> $(n^{[L-1]}, 209)$ </td>  <tr>
 <tr>
 <td> <b>couche L</b> </td>  <td> $(n^{[L]}, n^{[L-1]})$ </td>  <td> $(n^{[L]}, 1)$ </td>
 <td> $Z^{[L]} = W^{[L]} A^{[L-1]} + b^{[L]}$</td>
 <td> $(n^{[L]}, 209)$ </td>  <tr>
</table>

Remember that when vous calculer $W X + b$ in python, it carries out broadcasting. pour example, if: 
$$ W = \begin{bmatrix}
 w_{00} & w_{01} & w_{02} \\
 w_{10} & w_{11} & w_{12} \\
 w_{20} & w_{21} & w_{22} \end{bmatrix}\;\;\; X = \begin{bmatrix}
 x_{00} & x_{01} & x_{02} \\
 x_{10} & x_{11} & x_{12} \\
 x_{20} & x_{21} & x_{22} \end{bmatrix} \;\;\; b =\begin{bmatrix}
 b_0 \\
 b_1 \\
 b_2
\end{bmatrix}\tag{2}$$

Puis $WX + b$ allez be:

$$ WX + b = \begin{bmatrix}
 (w_{00}x_{00} + w_{01}x_{10} + w_{02}x_{20}) + b_0 & (w_{00}x_{01} + w_{01}x_{11} + w_{02}x_{21}) + b_0 & \cdots \\
 (w_{10}x_{00} + w_{11}x_{10} + w_{12}x_{20}) + b_1 & (w_{10}x_{01} + w_{11}x_{11} + w_{12}x_{21}) + b_1 & \cdots \\
 (w_{20}x_{00} + w_{21}x_{10} + w_{22}x_{20}) + b_2 & (w_{20}x_{01} + w_{21}x_{11} + w_{22}x_{21}) + b_2 & \cdots
\end{bmatrix}\tag{3} $$


<a name='ex-2'></a>
### Exercice 2 - initialize_parameters_deep

Implémentez initialisation pour unn Réseau de neurones profond à L couches. 
**instructions :**
- La structure du modèle is *[LINEAR -> RELU] $ \times$ (L-1) -> LINEAR -> SIGMOID*. I.e., it has $L-1$ layers en utilisant a ReLU activation fonction followed by an sortie couche avec a sigmoid activation fonction.
- Utilisez random initialisation pour le weight matrices. Utilisez `np.random.randn(d0, d1, ..., dn) * 0.01`.
- Utilisez zeros initialisation pour le biases. Utilisez `np.zeros(shape)`.
- Vous'll store $n^{[l]}$, le number of units in different layers, in a variable `layer_dims`. pour example, le `layer_dims` pour last week's Planar Data classification modèle would have been [2,4,1]: There were two inputs, one hidden couche avec 4 hidden units, et an sortie couche avec 1 sortie unit. This means `W1`'s shape was (4,2), `b1` was (4,1), `W2` was (1,4) et `b2` was (1,1). Now vous allez generalize this to $L$ layers! - Voici l'implémentation pour $L=1$ (réseau de neurones à une couche). Elle peut vous guider pour implémenter le cas général (réseau à L couches).
```python
 if L == 1:
 paramètres["W" + str(L)] = np.random.randn(layer_dims[1], layer_dims[0]) * 0.01
 paramètres["b" + str(L)] = np.zeros((layer_dims[1], 1))
```

In [ ]:
# FONCTION : initialize_parameters_deep
def initialize_parameters_deep(layer_dims):
    
    """ Arguments : layer_dims -- python array (list) containing le dimensions of each couche in notre network Retourne : paramètres -- python dictionary containing votre paramètres "W1", "b1", ..., "WL", "bL": Wl -- weight matrix of shape (layer_dims[l], layer_dims[l-1]) bl -- bias vector of shape (layer_dims[l], 1) """
    
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) # number of layers in le network
    for l in range(1, L):
        #(≈ 2 lines of code) # parameters['W' + str(l)] = ... # parameters['b' + str(l)] = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI assert(parameters['W' + str(l)].shape == (layer_dims[l], layer_dims[l - 1]))
        assert(parameters['b' + str(l)].shape == (layer_dims[l], 1))

        
    return parameters

In [ ]:
print("Test Case 1:\n")
parameters = initialize_parameters_deep([5,4,3])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

initialize_parameters_deep_test_1(initialize_parameters_deep)

print("\033[90m\nTest Case 2:\n")
parameters = initialize_parameters_deep([4,3,2])

print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))
initialize_parameters_deep_test_2(initialize_parameters_deep)

***Sortie attendue***
```
Cas de test 1:

W1 = [[ 0.01788628 0.0043651 0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865 0.00884622 0.00881318 0.01709573 0.00050034]
 [-0.00404677 -0.0054536 -0.01546477 0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565 0.01486148 0.00236716]
 [-0.01023785 -0.00712993 0.00625245 -0.00160513]
 [-0.00768836 -0.00230031 0.00745056 0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]
 All tests passed.

Cas de test 2:

W1 = [[ 0.01788628 0.0043651 0.00096497 -0.01863493]
 [-0.00277388 -0.00354759 -0.00082741 -0.00627001]
 [-0.00043818 -0.00477218 -0.01313865 0.00884622]]
b1 = [[0.]
 [0.]
 [0.]]
W2 = [[ 0.00881318 0.01709573 0.00050034]
 [-0.00404677 -0.0054536 -0.01546477]]
b2 = [[0.]
 [0.]]
 All tests passed.
```

<a name='4'></a>
## 4 - Module de propagation avant

<a name='4-1'></a>
### 4.1 - Propagation linéaire avant 
Maintenant que vous have initialiserd vos paramètres, vous can do le propagation avant module. Start by implementing some basic functions that vous can use again later when implementing le modèle. Now, vous'll complete three functions in this order:

- LINEAR
- LINEAR -> ACTIVATION where ACTIVATION allez be either ReLU or Sigmoid. - [LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID (modèle complet)

Le linear avant module (vectorized over all le examples) computes le following equations:

$$Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}\tag{4}$$

where $A^{[0]} = X$. 
<a name='ex-3'></a>
### Exercice 3 - linear_forward 
Construisez la partie linéaire de la propagation avant.

**Rappel :**
Le mathematical representation of this unit is $Z^{[l]} = W^{[l]}A^{[l-1]} +b^{[l]}$. Vous may also find `np.dot()` useful. If votre dimensions don't match, printing `W.shape` may help.

In [ ]:
# FONCTION : linear_forward
def linear_forward(A, W, b):
    """ Implémente la partie linéaire de la propagation avant d'une couche. Arguments : A -- activations from previous couche (or input data): (size of previous couche, number of examples) W -- weights matrix: numpy array of shape (size of current couche, size of previous couche) b -- bias vector, numpy array of shape (size of le current couche, 1) Retourne : Z -- le input of le activation fonction, also called pre-activation parameter cache -- a python tuple containing "A", "W" et "b" ; stored pour computing le arrière pass efficiently """
    
    #(≈ 1 line of code) # Z = ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI cache = (A, W, b) 
    return Z, cache

In [ ]:
t_A, t_W, t_b = linear_forward_test_case()
t_Z, t_linear_cache = linear_forward(t_A, t_W, t_b)
print("Z = " + str(t_Z))

linear_forward_test(linear_forward)

***Sortie attendue***
```
Z = [[ 3.26295337 -1.23429987]]
```

<a name='4-2'></a>
### 4.2 - Propagation linéaire-activation avant

Dans ce notebook, vous allez use two activation functions:

- **Sigmoid**: $\sigma(Z) = \sigma(W A + b) = \frac{1}{ 1 + e^{-(W A + b)}}$. Vous've been provided avec le `sigmoid` fonction which returns **two** items: le activation value "`a`" et a "`cache`" that contains "`Z`" (it's what we allez feed in to le corresponding arrière fonction). To use it vous could just call: ``` python
A, activation_cache = sigmoid(Z)
```

- **ReLU**: Le mathematical pourmula pour ReLu is $A = RELU(Z) = max(0, Z)$. Vous've been provided avec le `relu` fonction. This fonction returns **two** items: le activation value "`A`" et a "`cache`" that contains "`Z`" (it's what vous'll feed in to le corresponding arrière fonction). To use it vous could just call:
``` python
A, activation_cache = relu(Z)
```

Pour plus de simplicité, vous allez regrouper deux fonctions (Linéaire et Activation) en une seule fonction (LINÉAIRE->ACTIVATION). Vous implémenterez donc une fonction qui effectue l'étape de propagation avant LINÉAIRE, suivie de l'étape d'ACTIVATION.

<a name='ex-4'></a>
### Exercice 4 - linear_activation_forward

Implémentez propagation avant of le *LINEAR->ACTIVATION* couche. Mathematical relation is: $A^{[l]} = g(Z^{[l]}) = g(W^{[l]}A^{[l-1]} +b^{[l]})$ where le activation "g" can be sigmoid() or relu(). Utilisez `linear_forward()` et le correct activation fonction.

In [ ]:
# FONCTION : linear_activation_forward
def linear_activation_forward(A_prev, W, b, activation):
    """
 Implémente la propagation avant pour la couche LINÉAIRE->ACTIVATION

 Arguments :
 A_prev -- activations from previous couche (or input data): (size of previous couche, number of examples)
 W -- weights matrix: numpy array of shape (size of current couche, size of previous couche)
 b -- bias vector, numpy array of shape (size of le current couche, 1)
 activation -- le activation to be used in this couche, stored as a text string: "sigmoid" or "relu"

 Retourne :
 A -- le output of le activation fonction, also called le post-activation value 
 cache -- a python tuple containing "linear_cache" et "activation_cache";
 stored pour computing le arrière pass efficiently
 """
    
    if activation == "sigmoid":
        #(≈ 2 lines of code) # Z, linear_cache = ... # A, activation_cache = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    elif activation == "relu":
        #(≈ 2 lines of code) # Z, linear_cache = ... # A, activation_cache = ... # VOTRE CODE COMMENCE ICI
        # VOTRE CODE SE TERMINE ICI cache = (linear_cache, activation_cache)
    return A, cache

In [ ]:
t_A_prev, t_W, t_b = linear_activation_forward_test_case()

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "sigmoid")
print("Avec sigmoid : A = " + str(t_A))

t_A, t_linear_activation_cache = linear_activation_forward(t_A_prev, t_W, t_b, activation = "relu")
print("Avec ReLU : A = " + str(t_A))

linear_activation_forward_test(linear_activation_forward)

***Sortie attendue***
```
Avec sigmoid : A = [[0.96890023 0.11013289]]
Avec ReLU : A = [[3.43896131 0. ]]
```

**Note :** In deep learning, le "[LINEAR->ACTIVATION]" computation is counted as a single couche in le neural network, not two layers. 

<a name='4-3'></a>
### 4.3 - Modèle à L couches 
Pour encore plus de simplicité lors de l'implémentation du réseau à $L$ couches, vous aurez besoin d'une fonction qui répète la précédente (`linear_activation_forward` avec RELU) $L-1$ fois, puis enchaîne avec un `linear_activation_forward` utilisant SIGMOID.

<img src="images/model_architecture_kiank.png" style="width:600px;height:300px;">
<caption><center> <b>Figure 2</b> : *[LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID* modèle</center></caption><br>

<a name='ex-5'></a>
### Exercice 5 - L_model_forward

Implémentez propagation avant of le above modèle.

**instructions :** In le code below, le variable `AL` allez denote $A^{[L]} = \sigma(Z^{[L]}) = \sigma(W^{[L]} A^{[L-1]} + b^{[L]})$. (This is sometimes also called `Yhat`, i.e., this is $\hat{Y}$.) 
**Indices :**
- Utilisez le functions vous've previously written - Utilisez a pour loop to replicate [LINEAR->RELU] (L-1) times
- Don't pourget to keep track of le caches in le "caches" list. To add a new value `c` to a `list`, vous can use `list.append(c)`.

In [ ]:
# FONCTION : L_model_forward
def L_model_forward(X, parameters):
    """ Implémente la propagation avant pour [LINÉAIRE->RELU]*(L-1)->LINÉAIRE->SIGMOÏDE Arguments : X -- data, numpy array of shape (input size, number of examples) paramètres -- sortie of initialize_parameters_deep() Retourne : AL -- activation value from le sortie (last) couche caches -- list of caches containing: every cache of linear_activation_forward() (there are L of them, indexed from 0 to L-1) """

    caches = []
    A = X
    L = len(parameters) // 2                  # number of layers in le neural network
    
    # Implémenter [LINÉAIRE -> RELU]*(L-1). Ajouter "cache" à la liste "caches". # Le pour loop starts at 1 because couche 0 is le input pour l in range(1, L): A_prev = A 
        #(≈ 2 lines of code) # A, cache = ... # caches ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI # Implémenter LINÉAIRE -> SIGMOÏDE. Ajouter "cache" à la liste "caches". #(≈ 2 lines of code) # AL, cache = ... # caches ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI
    return AL, caches

In [ ]:
t_X, t_parameters = L_model_forward_test_case_2hidden()
t_AL, t_caches = L_model_forward(t_X, t_parameters)

print("AL = " + str(t_AL))

L_model_forward_test(L_model_forward)

***Sortie attendue***
```
AL = [[0.03921668 0.70498921 0.19734387 0.04728177]]
```

**génial!** Vous've implemented a full propagation avant that takes le input X et outputs a row vector $A^{[L]}$ containing votre predictions. It also records all intermediate values in "caches". en utilisant $A^{[L]}$, vous can calculer le coût of votre predictions.

<a name='5'></a>
## 5 - Fonction de coût

Vous pouvez maintenant implémenter la propagation avant et la rétropropagation. Vous devez calculer le coût afin de vérifier si votre modèle apprend réellement.

<a name='ex-6'></a>
### Exercice 6 - compute_cost
Calculez le coût d'entropie croisée $J$, en utilisant le following pourmula: $$-\frac{1}{m} \sum\limits_{i = 1}^{m} (y^{(i)}\log\left(a^{[L] (i)}\right) + (1-y^{(i)})\log\left(1- a^{[L](i)}\right)) \tag{7}$$


In [ ]:
# FONCTION : compute_cost
def compute_cost(AL, Y):
    """ Implémente la fonction de coût définie par l'équation (7). Arguments : AL -- probability vector corresponding to votre label predictions, shape (1, number of examples) Y -- true "label" vector (pour example: containing 0 if non-cat, 1 if cat), shape (1, number of examples) Retourne : coût -- cross-entropy coût """
    
    m = Y.shape[1]

    # Calculer loss from aL et y. # (≈ 1 lines of code) # coût = ... # VOTRE CODE COMMENCE ICI # VOTRE CODE SE TERMINE ICI coût = np.squeeze(coût) # To make sure votre coût's shape is what we expect (e.g. this turns [[17]] into 17).

    
    return cost

In [ ]:
t_Y, t_AL = compute_cost_test_case()
t_cost = compute_cost(t_AL, t_Y)

print("Cost: " + str(t_cost))

compute_cost_test(compute_cost)

**Sortie attendue :**
<table>
 <tr>
 <td><b>coût</b> </td>
 <td> 0.2797765635793422</td>  </tr>
</table>

<a name='6'></a>
## 6 - Module de rétropropagation

Comme pour la propagation avant, vous allez implémenter des fonctions auxiliaires pour la rétropropagation dans ce notebook. Rappelez-vous que la rétropropagation sert à calculer le gradient de la fonction de perte par rapport aux paramètres. 
**Rappel :** <img src="images/backprop_kiank.png" style="width:650px;height:250px;">
<caption><center><font color='purple'><b>Figure 3</b>: Avant et Rétropropagation pour LINEAR->RELU->LINEAR->SIGMOID <br> <i>Le purple blocks represent le propagation avant, et le red blocks represent le rétropropagation.</font></center></caption>


<!-- pour those of vous who are experts in calculus (which vous don't need to be to complete this notebook!), le chain rule of calculus can be used to derive le derivative of le loss $\mathcal{L}$ avec respect to $z^{[1]}$ in a 2-couche network as follows:

$$\frac{d \mathcal{L}(a^{[2]},y)}{{dz^{[1]}}} = \frac{d\mathcal{L}(a^{[2]},y)}{{da^{[2]}}}\frac{{da^{[2]}}}{{dz^{[2]}}}\frac{{dz^{[2]}}}{{da^{[1]}}}\frac{{da^{[1]}}}{{dz^{[1]}}} \tag{8} $$

Pour calculer le gradient $dW^{[1]} = \frac{\partial L}{\partial W^{[1]}}$, utilisez la règle de chaîne précédente : $dW^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial W^{[1]}}$. Pendant la rétropropagation, à chaque étape, vous multipliez votre gradient courant par le gradient de la couche correspondante pour obtenir le gradient recherché.

De manière équivalente, pour calculer le gradient $db^{[1]} = \frac{\partial L}{\partial b^{[1]}}$, vous utilisez la règle de chaîne précédente : $db^{[1]} = dz^{[1]} \times \frac{\partial z^{[1]} }{\partial b^{[1]}}$.

This is why we talk about **backpropagation**.
!-->

Now, similarly to propagation avant, vous're going pour construire le rétropropagation in three steps:
1. linear_backward
2. LINEAR -> ACTIVATION arrière where ACTIVATION computes le derivative of either le ReLU or sigmoid activation
3. [LINEAR -> RELU] $\times$ (L-1) -> LINEAR -> SIGMOID arrière (modèle complet)

Pour le prochain exercice, rappelez-vous que :

- `b` is a matrix(np.ndarray) avec 1 column et n rows, i.e: b = [[1.0], [2.0]] (remember that `b` is a constant)
- np.sum perpourms a sum over le elements of a ndarray
- axis=1 or axis=0 specify if le sum is carried out by rows or by columns respectively
- keepdims specifies if le original dimensions of le matrix must be kept.
- Look at le following example to clarify:

In [ ]:
A = np.array([[1, 2], [3, 4]])

print('axis=1 and keepdims=True')
print(np.sum(A, axis=1, keepdims=True))
print('axis=1 and keepdims=False')
print(np.sum(A, axis=1, keepdims=False))
print('axis=0 and keepdims=True')
print(np.sum(A, axis=0, keepdims=True))
print('axis=0 and keepdims=False')
print(np.sum(A, axis=0, keepdims=False))

<a name='6-1'></a>
### 6.1 - Propagation linéaire arrière

pour couche $l$, le linear part is: $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$ (followed by an activation).

Suppose vous have already calculated le derivative $dZ^{[l]} = \frac{\partial \mathcal{L} }{\partial Z^{[l]}}$. Vous want to get $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$.

<img src="images/linearback_kiank.png" style="width:250px;height:300px;">
<caption><center><font color='purple'><b>Figure 4</b></font></center></caption>

Le three outputs $(dW^{[l]}, db^{[l]}, dA^{[l-1]})$ are computed en utilisant le input $dZ^{[l]}$.

Here are le pourmulas vous need:
$$ dW^{[l]} = \frac{\partial \mathcal{J} }{\partial W^{[l]}} = \frac{1}{m} dZ^{[l]} A^{[l-1] T} \tag{8}$$
$$ db^{[l]} = \frac{\partial \mathcal{J} }{\partial b^{[l]}} = \frac{1}{m} \sum_{i = 1}^{m} dZ^{[l](i)}\tag{9}$$
$$ dA^{[l-1]} = \frac{\partial \mathcal{L} }{\partial A^{[l-1]}} = W^{[l] T} dZ^{[l]} \tag{10}$$


$A^{[l-1] T}$ is le transpose of $A^{[l-1]}$. 

<a name='ex-7'></a>
### Exercice 7 - linear_backward 
Utilise les 3 pourmules ci-dessus pour implémenter `linear_backward()`.

**Indice :**

- In numpy vous can get le transpose of an ndarray `A` en utilisant `A.T` or `A.transpose()`

In [ ]:
# FONCTION : linear_backward
def linear_backward(dZ, cache):
    """ Implémente la partie linéaire de la rétropropagation pour une seule couche (couche l) Arguments : dZ -- Gradient of le coût avec respect to le linear sortie (of current couche l) cache -- tuple of values (A_prev, W, b) coming from le propagation avant in le current couche Retourne : dA_prev -- Gradient of le coût avec respect to le activation (of le previous couche l-1), same shape as A_prev dW -- Gradient of le coût avec respect to W (current couche l), same shape as W db -- Gradient of le coût avec respect to b (current couche l), same shape as b """
    A_prev, W, b = cache
    m = A_prev.shape[1]

    ### DÉBUT DU CODE ### (≈ 3 lines of code) # dW = ... # db = ... sum by le rows of dZ avec keepdims=True # dA_prev = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI
    return dA_prev, dW, db

In [ ]:
t_dZ, t_linear_cache = linear_backward_test_case()
t_dA_prev, t_dW, t_db = linear_backward(t_dZ, t_linear_cache)

print("dA_prev: " + str(t_dA_prev))
print("dW: " + str(t_dW))
print("db: " + str(t_db))

linear_backward_test(linear_backward)

**Sortie attendue :**
```
dA_prev: [[-1.15171336 0.06718465 -0.3204696 2.09812712]
 [ 0.60345879 -3.72508701 5.81700741 -3.84326836]
 [-0.4319552 -1.30987417 1.72354705 0.05070578]
 [-0.38981415 0.60811244 -1.25938424 1.47191593]
 [-2.52214926 2.67882552 -0.67947465 1.48119548]]
dW: [[ 0.07313866 -0.0976715 -0.87585828 0.73763362 0.00785716]
 [ 0.85508818 0.37530413 -0.59912655 0.71278189 -0.58931808]
 [ 0.97913304 -0.24376494 -0.08839671 0.55151192 -0.10290907]]
db: [[-0.14713786]
 [-0.11313155]
 [-0.13209101]]
 ```

<a name='6-2'></a>
### 6.2 - Propagation linéaire-activation arrière

Ensuite, vous allez create a fonction that merges le two helper functions: **`linear_backward`** et le arrière step pour le activation **`linear_activation_backward`**. 
Pour vous aider à implémenter `linear_activation_backward`, deux fonctions de rétropropagation sont fournies :
- **`sigmoid_backward`** : implémente la rétropropagation pour l'unité SIGMOID. Vous pouvez l'appeler ainsi :

```python
dZ = sigmoid_backward(dA, activation_cache)
```

- **`relu_backward`** : implémente la rétropropagation pour l'unité RELU. Vous pouvez l'appeler ainsi :

```python
dZ = relu_backward(dA, activation_cache)
```

If $g(.)$ is le activation fonction, `sigmoid_backward` et `relu_backward` calculer $$dZ^{[l]} = dA^{[l]} * g'(Z^{[l]}). \tag{11}$$ 
<a name='ex-8'></a>
### Exercice 8 - linear_activation_backward

Implémentez backpropagation pour le *LINEAR->ACTIVATION* couche.

In [ ]:
# FONCTION : linear_activation_backward
def linear_activation_backward(dA, cache, activation):
    """
 Implémente la rétropropagation pour la couche LINÉAIRE->ACTIVATION.
 
 Arguments :
 dA -- post-activation gradient pour current couche l 
 cache -- tuple of values (linear_cache, activation_cache) we store pour computing rétropropagation efficiently
 activation -- le activation to be used in this couche, stored as a text string: "sigmoid" or "relu"
 
 Retourne :
 dA_prev -- Gradient of le coût avec respect to le activation (of le previous couche l-1), same shape as A_prev
 dW -- Gradient of le coût avec respect to W (current couche l), same shape as W
 db -- Gradient of le coût avec respect to b (current couche l), same shape as b
 """
    linear_cache, activation_cache = cache
    
    if activation == "relu":
        #(≈ 2 lines of code) # dZ = ... # dA_prev, dW, db = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    elif activation == "sigmoid":
        #(≈ 2 lines of code) # dZ = ... # dA_prev, dW, db = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    return dA_prev, dW, db

In [ ]:
t_dAL, t_linear_activation_cache = linear_activation_backward_test_case()

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "sigmoid")
print("Avec sigmoid : dA_prev = " + str(t_dA_prev))
print("Avec sigmoid : dW = " + str(t_dW))
print("Avec sigmoid : db = " + str(t_db))

t_dA_prev, t_dW, t_db = linear_activation_backward(t_dAL, t_linear_activation_cache, activation = "relu")
print("Avec relu : dA_prev = " + str(t_dA_prev))
print("Avec relu : dW = " + str(t_dW))
print("Avec relu : db = " + str(t_db))

linear_activation_backward_test(linear_activation_backward)

**Sortie attendue :**
```
Avec sigmoid : dA_prev = [[ 0.11017994 0.01105339]
 [ 0.09466817 0.00949723]
 [-0.05743092 -0.00576154]]
Avec sigmoid : dW = [[ 0.10266786 0.09778551 -0.01968084]]
Avec sigmoid : db = [[-0.05729622]]
Avec relu : dA_prev = [[ 0.44090989 0. ]
 [ 0.37883606 0. ]
 [-0.2298228 0. ]]
Avec relu : dW = [[ 0.44513824 0.37371418 -0.10478989]]
Avec relu : db = [[-0.20837892]]
```

<a name='6-3'></a>
### 6.3 - Rétropropagation du modèle à L couches 
Vous allez maintenant implémenter la fonction de rétropropagation pour l'ensemble du réseau ! 
Rappelez-vous que lorsque vous avez implémenté la fonction `L_model_forward`, à chaque itération vous avez stocké un cache contenant (X, W, b et z). Dans le module de rétropropagation, vous utiliserez ces variables pour calculer les gradients. Ainsi, dans la fonction `L_model_backward`, vous parcourrez toutes les couches cachées en sens inverse, en commençant par la couche $L$. À chaque étape, vous utiliserez les valeurs en cache de la couche $l$ pour rétropropager à travers cette couche. La Figure 5 ci-dessous illustre ce passage arrière. 

<img src="images/mn_backward.png" style="width:450px;height:300px;">
<caption><center><font color='purple'><b>Figure 5</b>: Arrière pass</font></center></caption>

**Initializing backpropagation**:

Pour rétropropager à travers ce réseau, vous savez que la sortie est : $A^{[L]} = \sigma(Z^{[L]})$. Votre code thus needs to calculer `dAL` $= \frac{\partial \mathcal{L}}{\partial A^{[L]}}$.
To do so, use this pourmula (derived en utilisant calculus which, again, vous don't need in-depth knowledge of!):
```python
dAL = - (np.divide(Y, AL) - np.divide(1 - Y, 1 - AL)) # derivative of coût avec respect to AL
```

Vous pouvez ensuite utiliser ce gradient post-activation `dAL` pour continuer la rétropropagation. Comme montré dans la Figure 5, vous pouvez injecter `dAL` dans la fonction arrière LINÉAIRE->SIGMOÏDE que vous avez implémentée (qui utilisera les valeurs mises en cache par la fonction `L_model_forward`). 
After that, vous allez have to use a `pour` loop to iterate through all le other layers en utilisant le LINEAR->RELU arrière fonction. Vous should store each dA, dW, et db in le grads dictionary. To do so, use this pourmula : 
$$grads["dW" + str(l)] = dW^{[l]}\tag{15} $$

pour example, pour $l=3$ this would store $dW^{[l]}$ in `grads["dW3"]`.

<a name='ex-9'></a>
### Exercice 9 - L_model_backward

Implémentez backpropagation pour le *[LINEAR->RELU] $\times$ (L-1) -> LINEAR -> SIGMOID* modèle.

In [ ]:
# FONCTION : L_model_backward
def L_model_backward(AL, Y, caches):
    """ Implémente la rétropropagation pour [LINÉAIRE->RELU] * (L-1) -> LINÉAIRE -> SIGMOÏDE Arguments : AL -- probability vector, sortie of le propagation avant (L_model_forward()) Y -- true "label" vector (containing 0 if non-cat, 1 if cat) caches -- list of caches containing: every cache of linear_activation_forward() avec "relu" (it's caches[l], pour l in range(L-1) i.e l = 0...L-2) le cache of linear_activation_forward() avec "sigmoid" (it's caches[L-1]) Retourne : grads -- A dictionary avec le gradients grads["dA" + str(l)] = ... grads["dW" + str(l)] = ... grads["db" + str(l)] = ... """
    grads = {}
    L = len(caches) # le number of layers
    m = AL.shape[1]
    Y = Y.reshape(AL.shape) # after this line, Y is le same shape as AL
    
    # Initializing le backpropagation #(1 line of code) # dAL = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI # Lth couche (SIGMOID -> LINEAR) gradients. Inputs: "dAL, current_cache". Outputs: "grads["dAL-1"], grads["dWL"], grads["dbL"] #(approx. 5 lines) # current_cache = ... # dA_prev_temp, dW_temp, db_temp = ... # grads["dA" + str(L-1)] = ... # grads["dW" + str(L)] = ... # grads["db" + str(L)] = ... # VOTRE CODE COMMENCE ICI 
    # VOTRE CODE SE TERMINE ICI # Loop from l=L-2 to l=0 pour l in reversed(range(L-1)): # lth couche: (RELU -> LINEAR) gradients. # Inputs: "grads["dA" + str(l + 1)], current_cache". Outputs: "grads["dA" + str(l)] , grads["dW" + str(l + 1)] , grads["db" + str(l + 1)] #(approx. 5 lines) # current_cache = ... # dA_prev_temp, dW_temp, db_temp = ... # grads["dA" + str(l)] = ... # grads["dW" + str(l + 1)] = ... # grads["db" + str(l + 1)] = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    return grads

In [ ]:
t_AL, t_Y_assess, t_caches = L_model_backward_test_case()
grads = L_model_backward(t_AL, t_Y_assess, t_caches)

print("dA0 = " + str(grads['dA0']))
print("dA1 = " + str(grads['dA1']))
print("dW1 = " + str(grads['dW1']))
print("dW2 = " + str(grads['dW2']))
print("db1 = " + str(grads['db1']))
print("db2 = " + str(grads['db2']))

L_model_backward_test(L_model_backward)

**Sortie attendue :**
```
dA0 = [[ 0. 0.52257901]
 [ 0. -0.3269206 ]
 [ 0. -0.32070404]
 [ 0. -0.74079187]]
dA1 = [[ 0.12913162 -0.44014127]
 [-0.14175655 0.48317296]
 [ 0.01663708 -0.05670698]]
dW1 = [[0.41010002 0.07807203 0.13798444 0.10502167]
 [0. 0. 0. 0. ]
 [0.05283652 0.01005865 0.01777766 0.0135308 ]]
dW2 = [[-0.39202432 -0.13325855 -0.04601089]]
db1 = [[-0.22007063]
 [ 0. ]
 [-0.02835349]]
db2 = [[0.15187861]]
```

<a name='6-4'></a>
### 6.4 - Mise à jour des paramètres

In this section, vous'll mettre à jour le paramètres of le modèle, avec la descente de gradient: 
$$ W^{[l]} = W^{[l]} - \alpha \text{ } dW^{[l]} \tag{16}$$
$$ b^{[l]} = b^{[l]} - \alpha \text{ } db^{[l]} \tag{17}$$

where $\alpha$ is le learning rate. 
After computing le updated paramètres, store them in le paramètres dictionary. 

<a name='ex-10'></a>
### Exercice 10 - Mettre à jour les paramètres

Implémentez `update_parameters()` pour mettre à jour vos paramètres avec la descente de gradient.

**instructions :**
Mettre à jour les paramètres par descente de gradient pour chaque $W^{[l]}$ et $b^{[l]}$, avec $l = 1, 2, ..., L$. 

In [ ]:
# FONCTION : update_parameters
def update_parameters(params, grads, learning_rate):
    """ Mettre à jour paramètres en utilisant gradient descent Arguments : params -- python dictionary containing votre paramètres grads -- python dictionary containing votre gradients, sortie of L_model_backward Retourne : paramètres -- python dictionary containing votre updated paramètres parameters["W" + str(l)] = ... parameters["b" + str(l)] = ... """
    parameters = copy.deepcopy(params)
    L = len(parameters) // 2 # number of layers in le neural network

    # Mettre à jour rule pour each parameter. Use a pour loop. #(≈ 2 lines of code) pour l in range(L): # parameters["W" + str(l+1)] = ... # parameters["b" + str(l+1)] = ... # VOTRE CODE COMMENCE ICI 
        # VOTRE CODE SE TERMINE ICI
    return paramètres

In [ ]:
t_parameters, grads = update_parameters_test_case()
t_parameters = update_parameters(t_parameters, grads, 0.1)

print ("W1 = "+ str(t_parameters["W1"]))
print ("b1 = "+ str(t_parameters["b1"]))
print ("W2 = "+ str(t_parameters["W2"]))
print ("b2 = "+ str(t_parameters["b2"]))

update_parameters_test(update_parameters)

**Sortie attendue :**
```
W1 = [[-0.59562069 -0.09991781 -2.14584584 1.82662008]
 [-1.76569676 -0.80627147 0.51115557 -1.18258802]
 [-1.0535704 -0.86128581 0.68284052 2.20374577]]
b1 = [[-0.04659241]
 [-1.28888275]
 [ 0.53405496]]
W2 = [[-0.55569196 0.0354055 1.32964895]]
b2 = [[-0.84610769]]
```

### Félicitations !

Vous venez d'implémenter toutes les fonctions nécessaires à la construction d'un réseau de neurones profond, notamment :

- L'utilisation d'unités non linéaires pour améliorer votre modèle
- La construction d'un réseau de neurones plus profond (avec plus d'une couche cachée)
- L'implémentation d'une classe de réseau de neurones simple d'utilisation

Dans le prochain notebook, vous combinerez toutes ces briques pour construire et entraîner deux modèles :

- Un réseau de neurones à deux couches
- Un réseau de neurones à L couches

Excellent travail !